# Is my model supported?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noelEOS/tensor-ee/blob/main/notebooks/02_is_my_model_supported.ipynb)

TensorEE runs a model only if every operation in it has a building block it can compute
exactly. This notebook checks a PyTorch model and says what is supported, what is refused, and
what to do next.

- **Fully offline:** no Earth Engine account, no requests, no downloads. The examples use
  random weights, which is all a support check needs: it reads the model's structure, not its
  predictions.
- **To run:** **Runtime → Run all**. Then put your own model in step 6.
- **Examples:** a supported U-Net; two refused models; an attention model through the converter.

## 1. Install

- The `swin` extra brings segmentation-models-pytorch 0.5.0 and timm 1.0.30, the versions the
  library was checked with.

In [ ]:
# Install from the repository: support for more layers is added there between releases.
%pip install --quiet "tensor-ee[swin] @ git+https://github.com/noelEOS/tensor-ee"

## 2. Setup

- The chip size the examples are checked at, and a helper that explains a report.

In [ ]:
INPUT_SHAPE = (1, 2, 64, 64)      # one chip: (1, bands, height, width)

In [ ]:
from collections import Counter

import torch
import segmentation_models_pytorch as smp

from tensor_ee import check_compatibility
from tensor_ee.smp import adapt_smp_model

torch.set_num_threads(2)
torch.manual_seed(0)              # random weights, the same every run


def explain(report):
    """Say in plain words what a compatibility report found."""
    facts = report.to_dict()
    if facts["status"] == "supported":
        storage = facts["weight_storage"]
        print("SUPPORTED: every operation in this model has a TensorEE building block.")
        print(f"  {facts['parameter_count']:,} parameters. After BatchNorm is folded into the "
              f"convolutions, {storage['coefficient_count']:,} numbers would be uploaded as weight assets.")
        print(f"  Output for one chip: {facts['output_shape']} (batch, classes, height, width).")
        return
    print(f"NOT SUPPORTED (status: {facts['status']}).")
    if not facts["evaluation"]["model_eval"]:
        print("  The model is in training mode: call model.eval() first.")
    if not facts["evaluation"]["float32_parameters_and_buffers"]:
        print("  Some weights are not float32: TensorEE computes in float32 at most.")
    if facts["trace"]["status"] != "success":
        print("  The model's code could not be traced:", facts["trace"]["error"])
    reasons = Counter((blocker["code"], blocker["message"]) for blocker in facts["blockers"])
    for (code, message), count in reasons.most_common(8):
        print(f"  {count} x {message}   [{code}]")
    if len(reasons) > 8:
        print(f"  ... and {len(reasons) - 8} more kinds of refusal (report.to_dict()['blockers'] lists them all)")
    if facts["unknowns"]:
        print(f"  {len(facts['unknowns'])} later operations were not checked, because they depend "
              "on the refused ones.")

## 3. A supported model: a U-Net with a ResNet-18 encoder

- The architecture of notebook 01's water model, with random weights.
- `adapt_smp_model` prepares an SMP model at a fixed chip size, and checks that it gives exactly
  the original's numbers on a test input.
- `check_compatibility` checks every operation.

In [ ]:
unet = smp.Unet("resnet18", encoder_weights=None, in_channels=2, classes=2).eval()
prepared = adapt_smp_model(unet, input_shape=INPUT_SHAPE)
print("Prepared model matches the original on a test input:",
      prepared.tensor_ee_smp_metadata["torch_equal"])
explain(check_compatibility(prepared, INPUT_SHAPE))

- **"Supported"** means TensorEE can express the model; it is not yet a claim about the numbers.
- Supported is not accurate: accuracy depends on training.

## 4. Refused models

### A LinkNet: transposed convolutions

- LinkNet's decoder upsamples with `ConvTranspose2d`, which has no CNN building block yet. Support for more layers is being added, so install from the repository (step 1) to get the latest.

In [ ]:
linknet = smp.Linknet("resnet18", encoder_weights=None, in_channels=2, classes=2).eval()
explain(check_compatibility(adapt_smp_model(linknet, input_shape=INPUT_SHAPE), INPUT_SHAPE))

### DeepLabV3+: dilated convolutions, and more

- Its encoder uses dilated convolutions; the CNN path supports stride 1 or 2, no dilation,
  "same" zero padding.
- The report also names its adaptive average pooling and bilinear resize.

In [ ]:
deeplab = smp.DeepLabV3Plus("resnet18", encoder_weights=None, in_channels=2, classes=2).eval()
explain(check_compatibility(adapt_smp_model(deeplab, input_shape=INPUT_SHAPE), INPUT_SHAPE))

**A refusal is firm:** TensorEE never approximates. It names the blocking operations; the ones
after them are not checked. Options:

- **Change the architecture** (for example a U-Net decoder instead of transposed convolutions)
  and retrain.
- **Ask for the operation:** [open an issue](https://github.com/noelEOS/tensor-ee/issues) with
  the message and your architecture. A new block comes with its own tests and evidence
  ([CONTRIBUTING](https://github.com/noelEOS/tensor-ee/blob/main/CONTRIBUTING.md)).

## 5. Attention models: the converter

- `adapt_smp_model` refuses transformer models and points to the converter,
  `tensor_ee.converter.convert`.
- The converter was built for SegFormer-B0 and a Swin-T UPerNet, on 224 × 224 chips.

In [ ]:
from tensor_ee.core.errors import UnsupportedOperation
from tensor_ee.core.grid import Grid
from tensor_ee.converter.convert import convert

segformer = smp.Segformer("mit_b0", encoder_weights=None, in_channels=2, classes=2).eval()
try:
    adapt_smp_model(segformer, input_shape=(1, 2, 224, 224))
except ValueError as refusal:
    print("adapt_smp_model:", refusal)

- Converting records the model's structure by running its code on placeholder tensors, so
  convert only models whose source you trust.
- The report gives each block a verdict: **inside** the settings validated in Earth Engine, or
  **outside**.

In [ ]:
chip = Grid("EPSG:3857", (2.5, 0, 1000, 0, -2.5, 2000), 224, 224)
b0 = convert(segformer, (1, 2, 224, 224), chip, name="segformer_b0")
print(b0.report.lines()[0])

- SegFormer-B0: all 18 blocks inside; the 2 layout operations only move pixels on and off the
  model's grid.
- SegFormer-B1 (same structure, wider) converts, but its blocks are **outside**: expressible,
  not yet validated. Outside is reported, not refused.

In [ ]:
segformer_b1 = smp.Segformer("mit_b1", encoder_weights=None, in_channels=2, classes=2).eval()
b1 = convert(segformer_b1, (1, 2, 224, 224), chip, name="segformer_b1")
print(b1.report.lines()[0])
print(next(node for node in b1.report.nodes if node.id == "encoder.block1.0").verdict())

- A refusal: SegFormer-B0 at 512 × 512 reduces its keys and values to more than one 7 × 7
  window, which needs a block the library does not have.

In [ ]:
try:
    convert(segformer, (1, 2, 512, 512), Grid("EPSG:3857", (2.5, 0, 1000, 0, -2.5, 2000), 512, 512))
except UnsupportedOperation as refusal:
    print("Refused:", refusal.code)
    print(str(refusal)[:400])

## 6. Your model

- Put your model in `MY_MODEL`, built exactly as trained (random weights give the same answer),
  and its chip shape in `MY_INPUT_SHAPE`. For example:

```python
MY_MODEL = smp.Unet("resnet34", encoder_weights=None, in_channels=4, classes=3)
MY_MODEL.load_state_dict(torch.load("my_checkpoint.pth", map_location="cpu", weights_only=True))
MY_INPUT_SHAPE = (1, 4, 256, 256)
```

- The check tries the CNN path (SMP or plain PyTorch CNN), then the converter.

In [ ]:
MY_MODEL = None                      # your torch.nn.Module
MY_INPUT_SHAPE = (1, 2, 224, 224)    # (1, bands, height, width)

In [ ]:
from tensor_ee.core.errors import UnsupportedOperation


def check_my_model(model, input_shape):
    """Try the CNN path, then, if that does not support the model, the converter."""
    model = model.eval()
    print("CNN path (adapt_smp_model and check_compatibility):")
    try:
        if type(model).__module__.startswith("segmentation_models_pytorch"):
            model_for_check = adapt_smp_model(model, input_shape=input_shape)
        else:
            model_for_check = model      # a plain PyTorch CNN is checked as it is
        report = check_compatibility(model_for_check, input_shape)
        explain(report)
        if report.status == "supported":
            return report
    except (ValueError, RuntimeError, UnsupportedOperation) as refusal:
        print("NOT SUPPORTED:", refusal)

    print()
    print("Attention-model path (the converter):")
    height, width = input_shape[2], input_shape[3]
    try:
        try:
            result = convert(model, input_shape, Grid("EPSG:3857", (2.5, 0, 1000, 0, -2.5, 2000),
                                                      width, height), name="my_model")
        except UnsupportedOperation as refusal:
            if "source_grid=None" not in str(refusal):
                raise
            result = convert(model, input_shape, name="my_model")   # a Swin encoder starts on its own grid
        print(result.report.lines()[0])
        print("Every block's verdict: print(MY_RESULT.report.text())")
        return result
    except UnsupportedOperation as refusal:
        print("NOT SUPPORTED:", refusal.code, "-", str(refusal)[:600])
    except Exception as failure:      # for example, torch.export could not trace the model
        print("NOT SUPPORTED: the converter could not read this model:", type(failure).__name__, failure)


if MY_MODEL is None:
    print("Put your model in MY_MODEL in the cell above, then run this cell again.")
else:
    MY_RESULT = check_my_model(MY_MODEL, MY_INPUT_SHAPE)

## What next

- **Supported CNN:** [notebook 03](https://github.com/noelEOS/tensor-ee/blob/main/notebooks/03_bring_your_own_cnn.ipynb)
  takes it from checkpoint to an exported region.
- **Converted attention model:** the
  [converter guide](https://github.com/noelEOS/tensor-ee/blob/main/docs/converter.md) plans and
  runs its stages.
- **Outside the validated records:** run it and compare with PyTorch on your input to make that
  evidence; the maintainer can advise.
- **Refused:** see step 4. The
  [block catalogue](https://github.com/noelEOS/tensor-ee/blob/main/docs/building-blocks.md)
  lists every block and its validated settings.